In [ ]:
!pip install -U transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 54.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset #Permette di importare dataset (anche direttamente da hugging face), evita di caricare tutto il dataset in RAM e implementa trasformazioni efficenti. il trainer di hugging face si aspetta dati in formato Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, TrainerCallback
import evaluate #Libreria di hugging face per metriche standard di valutazione dell'output
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


print(torch.cuda.is_available())

model_name = "bert-base-uncased"
ds = load_dataset("dair-ai/emotion", "split")
ROOT_RESULT = "./results/Graphs/"

training_set = ds['train']
validation_set = ds['validation']
test_set = ds['test']

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_C_FT = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
model_L_FT = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
#Per risultati più attendibili utilizziamo due modelli, uno per il light e uno per il complete fine-tuning
#Eseguire il FT completo sul modello light fine-tuned potrebbe alterare i risultati

model_performance = {} #Raccoglie i risultati

def plot_confusion_matrix_func(predictions, true_P, filename):
    """
    Crea e salva una confusion matrix.

    Parameters:
        predictions (np.array): array delle previsioni del modello.
        labels (np.array): array delle etichette reali.
        class_names (list): lista dei nomi delle classi.
        filename (str): nome del file immagine da salvare.
    """
    cm = confusion_matrix(true_P, predictions)

    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues, ax=ax)

    ax.set_title(f"Confusion Matrix ({filename})")
    plt.savefig(ROOT_RESULT + f"Confusion_Matrix_{filename}.png", dpi=600, bbox_inches="tight")
    plt.close()
    print(f"Confusion matrix salvata come Confusion_Matrix_{filename}.png")

def evaluate_inContext(trainer, test_set_inContext_Learning, fine_tuned = False):
    partial_results = {}
    for test_inContext in test_set_inContext_Learning:
        print(f"========== Evaluetion test-set with in-context ({test_inContext}) {'after' if fine_tuned else 'before'} complete fine-tuning ==========")
        partial_results[test_inContext] = trainer.evaluate(test_set_inContext_Learning[test_inContext])

        predictions_L_FT = trainer.predict(test_set_inContext_Learning[test_inContext]) #Preleva le previsioni del modello fine-tuned sul test set, restituisce un oggetto con vari campi, tra cui "predictions" che contiene i logits (output grezzo del modello prima della softmax) per ogni esempio del test set
        logits_L_FT = predictions_L_FT.predictions #I logits sono un array di dimensione (num_examples, num_labels) che contiene i punteggi grezzi per ogni classe. Per ottenere le classi predette, si applica argmax sui logits lungo l'asse delle classi (axis=-1) per selezionare la classe con il punteggio più alto per ogni esempio
        predicted_labels_L_FT = np.argmax(logits_L_FT, axis=-1)
        plot_confusion_matrix_func(predicted_labels_L_FT, test_set_inContext_Learning[test_inContext]['label'], f"Light_Fine-Tuning_{test_inContext}")

    return partial_results

def plot_class_metrics(eval_results, class_names=None, filename="class_metrics.png"):
    """
    Crea e salva un grafico a barre raggruppate (Precision, Recall, F1)
    per ciascuna classe.

    Parameters:
        eval_results (dict): dizionario contenente
            - "eval_precision"
            - "eval_recall"
            - "eval_f1"
        class_names (list, optional): nomi delle classi
        filename (str): nome file immagine da salvare
    """

    precision = eval_results["eval_precision"]
    recall = eval_results["eval_recall"]
    f1 = eval_results["eval_f1"]

    n_classes = len(precision)
    x = np.arange(n_classes)
    width = 0.25

    # Se non vengono passati nomi classe, usa numeri
    if class_names is None:
        class_names = [f"Class {i}" for i in range(n_classes)]

    plt.figure(figsize=(12,7))

    bars1 = plt.bar(x - width, precision, width=width, label="Precision")
    bars2 = plt.bar(x, recall, width=width, label="Recall")
    bars3 = plt.bar(x + width, f1, width=width, label="F1")

    plt.xticks(x, class_names)
    plt.xlabel("Classes")
    plt.ylabel("Performance")
    plt.title(f"Precision, Recall, F1 per Class: {filename}")
    plt.legend()
    plt.ylim(0, 1)  # metriche tra 0 e 1

    # Aggiunge valore sopra ogni barra
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width()/2,
                height + 0.01,
                f"{height:.2f}",
                ha='center',
                va='bottom',
                fontsize=8
            )

    plt.savefig(ROOT_RESULT + filename, dpi=600, bbox_inches="tight")
    plt.close()

    print(f"Grafico salvato come {filename}")

class LiveLossPlotCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.train_epochs = []
        self.train_losses = []
        self.grad_norm = []
        self.eval_epochs = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        if "loss" in logs and "epoch" in logs and "grad_norm" in logs:
            self.train_epochs.append(logs["epoch"])
            self.train_losses.append(logs["loss"])
            self.grad_norm.append(logs["grad_norm"])

        if "eval_loss" in logs and "epoch" in logs:
            self.eval_epochs.append(logs["epoch"])
            self.eval_losses.append(logs["eval_loss"])

        self.update_plot()

    def update_plot(self):

        # ===== GRAFICO GRAD NORM =====
        plt.figure(figsize=(12, 7))
        plt.plot(self.train_epochs, self.grad_norm)

        plt.xlabel("Epoche")
        plt.ylabel("Grad_norm")
        plt.title(f"Norma L2 dei gradienti (Live): {self.name}")
        plt.grid(True)

        plt.savefig(f"{ROOT_RESULT + self.name + '_Grad_norm'}.png", dpi=600, bbox_inches="tight")
        plt.close()

        # ===== GRAFICO LOSS =====
        plt.figure(figsize=(12, 7))

        plt.plot(self.train_epochs, self.train_losses)
        plt.plot(self.eval_epochs, self.eval_losses)

        plt.xlabel("Epoche")
        plt.ylabel("Loss")
        plt.title(f"Training Loss vs Validation Loss (Live): {self.name}")
        plt.legend(["Training Loss", "Validation Loss"])
        plt.grid(True)

        # Calcolo minimo validation loss
        if len(self.eval_losses) > 0:
            min_loss = min(self.eval_losses)
            min_epoch = self.eval_epochs[self.eval_losses.index(min_loss)]

            text = f"Minimum Validation Loss: {min_loss:.4f} at Epoch {min_epoch:.2f}"

            plt.figtext(
                0.5, 0.01,
                text,
                ha="center",
                fontsize=10,
                bbox=dict(facecolor='white', alpha=0.6)
            )

        plt.savefig(f"{ROOT_RESULT + self.name}.png", dpi=600, bbox_inches="tight")
        plt.close()


def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)
    # Il tokenizer prende tre parametri, il primo è l'esempio mentre il secondo e il terzo stabiliscono cosa fare con esempi più brevi e più lunghi della dimensione massima dell'input
    # Se l'imput è minore dei token massimi allora con "padding="max_length" vengono aggiunti dei token di padding che vengono ignorati nella fase di self-attention
    # Se l'input supera i token massimi con "truncation=True" allora la frase viene troncata. Si può specificare quanti token accettare, di default max_length

one_shot = """
Classify the sentence on base the emotions author:

i feel empty and exhausted, like nothing i do really matters anymore => 0
"""

three_shot = one_shot + """
today i woke up smiling because everything finally seems to be going right => 1

i feel deeply connected to you and grateful to have you in my life => 2
"""

five_shot = one_shot + three_shot + """
it makes me furious when people ignore my efforts and take advantage of me => 3

my heart started racing when i realized i had lost my passport => 4
"""

test_set_inContext_Learning = {
    "one_shot" : ds['test'].map( lambda x : {'text' : (one_shot + x['text'] + ' =>'), 'label' : x['label']}).map(tokenize, batched=True),
    "three_shot" : ds['test'].map( lambda x : {'text' : (three_shot + x['text'] + ' =>'), 'label' : x['label']}).map(tokenize, batched=True),
    "five_shot" : ds['test'].map( lambda x : {'text' : (five_shot + x['text'] + ' =>'), 'label' : x['label']}).map(tokenize, batched=True)
    }

training_set = training_set.map(tokenize, batched=True)# Viene applicata la funzione di tokenizzazione al dataset. Il parametro batched=True permette di non tokenizzare una esempio per volta, ma divide il dataset in batch e verranno tokenizzati in parallelo
validation_set = validation_set.map(tokenize, batched=True)
test_set = test_set.map(tokenize, batched=True)

#Evaluate è un oggetto che contiene il metodo .compute() che implementa una versione di confronto tra la prediction e il label
metric_accuracy = evaluate.load("accuracy")
#Right prediction / Whole predictions
metric_recall = evaluate.load("recall")
#Per ogni label I => True label I / Whole true label I
metric_precision = evaluate.load("precision")
#Per ogni label I => True label I / True label I + False Label I
metric_f1 = evaluate.load("f1")
# F1 = 2*(precision*recall / precision+recall), Utile per dataset sbilanciato

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    #Logits è l'output grezzo del modello prima della softmax (linear classifier)
    predictions = np.argmax(logits, axis=-1)
    # convert the logits to their predicted class

    # Accuracy
    acc = metric_accuracy.compute(
        predictions=predictions,
        references=labels
    )

    # Recall macro
    recall = metric_recall.compute(
        predictions=predictions,
        references=labels,
        average=None
        #average = macro => media aritmetica delle recall. None => recall calcolata per ogni classe e ritorna un vettore. weighted => media pesata tra le recall
    )

    precision = metric_precision.compute(
        predictions=predictions,
        references=labels,
        average=None
    )

    # F1 macro
    f1 = metric_f1.compute(
        predictions=predictions,
        references=labels,
        average=None
    )

    return {
        "accuracy": acc["accuracy"],
        "recall": recall["recall"].tolist(),
        "precision": precision["precision"].tolist(),
        "f1": f1["f1"].tolist()
    }

training_args_light_FT = TrainingArguments(
    output_dir="./results/Light_fine-tuning",

    num_train_epochs=14,                 # Numero di epoche. Epoche -> numero di volte che il modello rivede gli stessi dati per addestrarsi

    eval_strategy="steps",              # Valutaazione del training ogni step
    eval_steps=250,                     # Ogni quanti step eseguire la valutazione. Step = numero di betch processati

    #logging_strategy="steps",
    logging_steps=250,                    # Ogni quanti step salvare le metriche di training
    logging_first_step=True,            # Mostra le metriche di training appena inizia l'addestramento (Il punto di partenza)

    learning_rate=4e-3,                 # Il tasso di apprendimento iniziale per l'ottimizzatore, determina la dimensione dei passi durante l'aggiornamento dei pesi del modello. 1*10^-3
    weight_decay=0.01,                  # Il fattore di decadimento di peso. Tira verso 0 i valori, più il valore è grande e più lo penalizza, limita l'overfitting

    per_device_train_batch_size=16,     # Il numero di esempi di training processati in parallelo per GPU/CPU durante l'addestramento.
    #per_device_eval_batch_size=32,      # parallelo durante il validation

    greater_is_better=True,             # Prende il modello migliore nella validazione

    fp16=True                           # Abilita l'addestramento in fp16 se supportato
)

training_args_complete_FT = TrainingArguments(
    output_dir="./results/Complete_fine-tuning",

    num_train_epochs=6,                 # Numero di epoche. Epoche -> numero di volte che il modello rivede gli stessi dati per addestrarsi

    eval_strategy="steps",              # Valutaazione alle fine di ogni epoca
    eval_steps=250,                     # Ogni quanti step eseguire la valutazione. Step = numero di betch processati

    #logging_strategy="steps",
    logging_steps=250,                    # Ogni quanti step salvare le metriche di training
    logging_first_step=True,            # Mostra le metriche di training appena inizia l'addestramento (Il punto di partenza)

    learning_rate=2e-5,                 # Il tasso di apprendimento iniziale per l'ottimizzatore, determina la dimensione dei passi durante l'aggiornamento dei pesi del modello. 1*10^-3
    weight_decay=0.001,                  # Il fattore di decadimento di peso. Tira verso 0 i valori, più il valore è grande e più lo penalizza, limita l'overfitting. Con betch size piccole, conviene utilizzare un fattore di decadimento minore, questo perché con pochi esempi (Es. 16) il rumore è maggiore

    per_device_train_batch_size=16,     # Il numero di esempi di training processati in parallelo per GPU/CPU durante l'addestramento.
    #per_device_eval_batch_size=32,      # parallelo durante il validation

    greater_is_better=True,             # Prende il modello migliore nella validazione

    fp16=True                           # Abilita l'addestramento in fp16 se supportato
)

liveTrack_Light_FT = LiveLossPlotCallback("Align_Classifier")

trainer_L_FT = Trainer(
    model=model_L_FT,
    args=training_args_light_FT,
    train_dataset=training_set.train_test_split(train_size=0.35, seed=42)["train"], # training_set.train_test_split divide il training_set in due "train" e "test" in cui il 15% dei dati vanno al train. La scelta dei simple è casuale con seed=42
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
    callbacks=[liveTrack_Light_FT]
)

liveTrack_Complete_FT = LiveLossPlotCallback("Complete_FT")

#Nuovo trainer necessario per ricreare da zero lo stato interno del trainer. Valori come il larning rate (Alterati dalla sua policy di gestione) non vengono re-inizializzati
trainer_C_FT = Trainer(
    model=model_C_FT,
    args=training_args_complete_FT,
    train_dataset=training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
    callbacks=[liveTrack_Complete_FT]
)

# Alignment classifier to task
for layers in model_L_FT.parameters():
    layers.requires_grad = False

for layers in model_L_FT.classifier.parameters():
    layers.requires_grad = True

trainer_L_FT.train() #Light fine-tuning, addestra solo il classification head per allineare il modello al task

model_performance["inContext_Light_fine-tuning"] = evaluate_inContext(trainer_L_FT, test_set_inContext_Learning,)
for grade_name, grade_values in model_performance["inContext_Light_fine-tuning"].items():
    plot_class_metrics(
        eval_results=grade_values,
        filename=f"{grade_name}.png"
    )

torch.cuda.empty_cache()
torch.cuda.synchronize()

trainer_C_FT.train() #Fine-tuning completo

#model_performance["inContext_Complete_fine-tuning"] = evaluate_inContext(trainer_C_FT, test_set_inContext_Learning, True)

model_performance["model_fine-tuned"] = trainer_C_FT.evaluate(test_set)
plot_class_metrics(model_performance["model_fine-tuned"],filename="Fine-tuned")

# Save evaluation results to a JSON file
with open("resultEvaluation.json", "w") as f:
    json.dump(model_performance, f, indent=4)

predictions_C_FT = trainer_C_FT.predict(test_set)
logits_C_FT = predictions_C_FT.predictions
predicted_labels_C_FT = np.argmax(logits_C_FT, axis=-1)
plot_confusion_matrix_func(predicted_labels_C_FT, test_set['label'], "Complete_Fine-Tuning")



True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Confusion matrix salvata come Confusion_Matrix_Light_Fine-Tuning_three_shot.png
========== Evaluetion test-set with in-context (five_shot) before complete fine-tuning ==========


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Confusion matrix salvata come Confusion_Matrix_Light_Fine-Tuning_five_shot.png
Grafico salvato come one_shot.png
Grafico salvato come three_shot.png
Grafico salvato come five_shot.png


Step,Training Loss,Validation Loss,Accuracy,Recall,Precision,F1
250,1.031108,0.519859,0.830000,"[0.9254545454545454, 0.9417613636363636, 0.5393258426966292, 0.7454545454545455, 0.8443396226415094, 0.09876543209876543]","[0.8540268456375839, 0.8478260869565217, 0.897196261682243, 0.8760683760683761, 0.6580882352941176, 0.8888888888888888]","[0.8883071553228621, 0.892328398384926, 0.6736842105263158, 0.8055009823182712, 0.7396694214876033, 0.17777777777777778]"
